# 통합 마케팅 (Marketing) 전략

# 데이터 기반 통합 이탈 방어 및 ROI 극대화 전략
> **목적**: 3대 모델(강도/빈도/습관)의 결과를 통합하여 마케팅 비용 대비 방어 이익이 가장 높은 '설득 가능군' 식별 및 수익 레버리지 시뮬레이션

### 주요 분석 스토리라인:
1. **데이터 통합 및 매핑**: 3개 주제별 모델의 이탈 확률을 병합하여 고객별 통합 위험 지수 및 8개 세그먼트(S~C 그룹) **정의**
2. **수익 중심 분석**: 
   - **Targeted Profit Curve**: 마케팅 비용과 CLV를 고려하여 순이익이 최대화되는 최적 임계치(Top 9.1%) **도출**
   - **전략 비교**: 공격적 전략(전체 대상) 대비 안정적 전략(정예 타겟)의 재무적 우위 **규명**
3. **ROI 시뮬레이션**: 방어 성공률(10%~20%) 상승 시나리오에 따른 수익 레버리지(2.9배) **확인**
4. **실행 로드맵 수립**: 안정군 대상 매몰 비용 절감 및 고효율 세그먼트 집중 투입을 통한 ROI 극대화 전략 **수립**
5. **비즈니스 가치 증명**: 데이터 과학 모델이 실제 재무 성과로 연결되는 프로세스의 타당성 **확보**

In [ ]:
# STEP 0. 환경 설정 및 데이터 통합

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

print("STEP 0: 마케팅 분석 데이터 통합 시작")

# 1. 각 주제별 결과 로드 (이전 단계 저장 파일명 기준)
res_intensity = pd.read_csv('result_churn_intensity.csv')
res_frequency = pd.read_csv('result_churn_frequency.csv')
res_habit = pd.read_csv('result_churn_habit.csv')

# 2. 통합 마케팅 데이터프레임 구축 (발급회원번호 기준 병합)
user_master = res_intensity[['발급회원번호', 'is_churn', 'intensity_prob']].copy()
user_master = user_master.merge(res_frequency[['발급회원번호', 'frequency_prob']], on='발급회원번호')
user_master = user_master.merge(res_habit[['발급회원번호', 'habit_prob']], on='발급회원번호')

# 3. 통합 위험 지수 (Total Risk Score) 산출
user_master['total_risk_score'] = user_master[['intensity_prob', 'frequency_prob', 'habit_prob']].mean(axis=1)

In [ ]:
# STEP 1. 통합 기대이익곡선 (Targeted Profit Curve) 도출

def plot_profit_curve(df):
    print("STEP 1: 통합 기대이익곡선 분석 수행 중...")
    
    # PPT 비즈니스 상수 설정
    unit_cost = 5000       # 1인당 마케팅 오퍼 비용
    unit_revenue = 150000  # 방어 성공 시 기대 매출 (CLV)
    success_rate = 0.10    # 기본 반응 성공률 10%
    
    df_sorted = df.sort_values('total_risk_score', ascending=False).reset_index(drop=True)
    thresholds = np.linspace(0, 1, 100)
    profits = []
    
    for t in thresholds:
        targets = df_sorted[df_sorted['total_risk_score'] >= t]
        n_targets = len(targets)
        true_positives = targets['is_churn'].sum()
        
        # 순이익 = (방어 성공 인원 * 매출) - (투입 인원 * 비용)
        profit = (true_positives * success_rate * unit_revenue) - (n_targets * unit_cost)
        profits.append(profit)
    
    # 시각화
    plt.figure(figsize=(10, 5))
    plt.plot(thresholds, profits, color='royalblue', lw=3)
    plt.axvline(thresholds[np.argmax(profits)], color='red', linestyle='--', label=f'Best Threshold: {thresholds[np.argmax(profits)]:.2f}')
    plt.title('통합 기대이익곡선 (Profit Curve)', fontsize=14)
    plt.xlabel('이탈 위험 지수 임계치')
    plt.ylabel('예상 순이익 (원)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_profit_curve(user_master)

In [ ]:
# STEP 2. 공격적 전략 vs 안정적 전략 성과 비교

def compare_strategies(df):
    print("STEP 2: 마케팅 전략 시나리오 비교 분석...")
    
    cost_head = 5000
    rev_head = 150000
    s_rate = 0.10
    
    # 1. 공격적 전략 (전체 위험군 대상)
    agg_target = df[df['total_risk_score'] >= 0.1]
    n_agg = len(agg_target)
    agg_profit = (agg_target['is_churn'].sum() * s_rate * rev_head) - (n_agg * cost_head)
    
    # 2. 안정적 전략 (최적 임계치 대상 - 상위 9.1%)
    stable_threshold = df['total_risk_score'].quantile(0.909)
    stable_target = df[df['total_risk_score'] >= stable_threshold]
    n_stable = len(stable_target)
    stable_profit = (stable_target['is_churn'].sum() * s_rate * rev_head) - (n_stable * cost_head)
    
    comparison = pd.DataFrame({
        '전략 구분': ['공격적 전략', '안정적 전략'],
        '타겟 인원': [n_agg, n_stable],
        '순수 방어 이익(억)': [agg_profit/1e8, stable_profit/1e8],
        'ROI': [agg_profit/(n_agg*cost_head), stable_profit/(n_stable*cost_head)]
    })
    
    display(comparison)
    return comparison

strategy_res = compare_strategies(user_master)

In [ ]:
# STEP 3. 8개 세그먼트 전략 매핑 (S~C 그룹)

print("STEP 3: 3대 지표 조합 기반 8개 세그먼트 매핑...")

# 상위 20%를 각 지표의 위험군(1)으로 정의
q = 0.8
user_master['I_risk'] = (user_master['intensity_prob'] >= user_master['intensity_prob'].quantile(q)).astype(int)
user_master['F_risk'] = (user_master['frequency_prob'] >= user_master['frequency_prob'].quantile(q)).astype(int)
user_master['H_risk'] = (user_master['habit_prob'] >= user_master['habit_prob'].quantile(q)).astype(int)

# 세그먼트 코드 생성 (I-F-H)
user_master['seg_code'] = user_master['I_risk'].astype(str) + user_master['F_risk'].astype(str) + user_master['H_risk'].astype(str)

seg_map = {
    '111': 'S그룹(긴급대응)', '110': 'A그룹(VIP유지_IF)', '101': 'A그룹(VIP유지_IH)',
    '011': 'B그룹(일반리텐션)', '100': 'A그룹(VIP유지_I)', '010': 'B그룹(일반리텐션_F)',
    '001': 'C그룹(조기예방)', '000': '안정군(Safe)'
}
user_master['segment'] = user_master['seg_code'].map(seg_map)

In [ ]:
# STEP 4. 수익 레버리지 시뮬레이션 (최종 결론)

print("STEP 4: 방어 성공률 상승 시나리오 시뮬레이션...")

# 안정적 전략 타겟 대상
stable_threshold = user_master['total_risk_score'].quantile(0.909)
final_targets = user_master[user_master['total_risk_score'] >= stable_threshold]
total_cost = len(final_targets) * 5000
actual_churners = final_targets['is_churn'].sum()

leverage_results = []
for rate in [0.10, 0.15, 0.20]:
    revenue = actual_churners * rate * 150000
    profit = revenue - total_cost
    leverage_results.append({'성공률': f'{rate*100:.0f}%', '순이익(억)': profit/1e8, 'ROI': profit/total_cost})

leverage_df = pd.DataFrame(leverage_results)
print(leverage_df)

# 최종 타겟 리스트 저장
user_master.to_csv('final_marketing_target_list.csv', index=False, encoding='utf-8-sig')
print("✅ 마케팅 타겟 리스트 저장 완료.")